In [ ]:
import sys, json, random, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'  GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('  ⚠️  No GPU — switch runtime before training')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT  = Path('/content/drive/MyDrive/CropClassifier_v2')
VERSION     = 'v2'
BASE        = Path('/content/dataset/combined_resized/Volumes/STORM 1100X/combined')  # same source as Stage 1
STAGE1_DEDUP_DIR = DRIVE_ROOT / f'dedup_checkpoints_{VERSION}'  # reuse Stage 1's cleaned image lists

MODEL_DIR   = DRIVE_ROOT / 'models' / f'stage2_disease_{VERSION}'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME       = 'efficientnet_b0'
NUM_CLASSES      = None   # set after manifest build
IMG_SIZE         = 224
NORM_MEAN        = (0.5, 0.5, 0.5)
NORM_STD         = (0.5, 0.5, 0.5)

BATCH_SIZE       = 64
NUM_WORKERS      = 4
DROPOUT          = 0.3
LABEL_SMOOTHING  = 0.1
EPOCHS           = 20
LR               = 1e-3
WEIGHT_DECAY     = 1e-4
MIN_DISEASE_COUNT = 20   # warn-only floor — nothing gets dropped, matches old notebook behavior

print(f'\n✅  Setup complete')
print(f'    Stage        : 2 — Disease Classification (v2, all 33 crops)')
print(f'    Model        : {MODEL_NAME}')
print(f'    Reusing Stage 1 cleaned data from: {STAGE1_DEDUP_DIR}')

Device : cuda
  GPU  : NVIDIA L4
  VRAM : 23.7 GB
Mounted at /content/drive

✅  Setup complete
    Stage        : 2 — Disease Classification (v2, all 33 crops)
    Model        : efficientnet_b0
    Reusing Stage 1 cleaned data from: /content/drive/MyDrive/CropClassifier_v2/dedup_checkpoints_v2


In [ ]:
import subprocess

if not BASE.exists() or sum(1 for p in BASE.iterdir() if p.is_dir()) < 10:
    print("Extracting combined_resized from Drive (split zip)...")
    COMBINED_SRC = Path('/content/drive/MyDrive/disease_detection/combined_resized')
    LOCAL_ZIP_DIR = Path('/content/combined_zip_parts')
    LOCAL_ZIP_DIR.mkdir(parents=True, exist_ok=True)

    if not (LOCAL_ZIP_DIR / 'combined.zip').exists():
        subprocess.run(f"cp {COMBINED_SRC}/combined.z0* {COMBINED_SRC}/combined.zip {LOCAL_ZIP_DIR}/",
                        shell=True, check=True)
        print("✅ Copied zip parts to local disk")
    else:
        print("✅ Zip parts already local, skipping copy")

    subprocess.run("which zip || apt-get -qq install -y zip", shell=True, check=True)

    FULL_ZIP = LOCAL_ZIP_DIR / 'combined_full.zip'
    if not FULL_ZIP.exists():
        print("Recombining split zip parts...")
        result = subprocess.run(
            f"cd {LOCAL_ZIP_DIR} && zip -s 0 combined.zip --out combined_full.zip",
            shell=True, capture_output=True, text=True
        )
        if result.returncode != 0:
            print("❌ STDERR:", result.stderr[-2000:])
            raise RuntimeError("Recombine failed — check output above")
        print("✅ Recombined")
    else:
        print("✅ Recombined zip already exists, skipping")

    Path("/content/dataset/combined_resized").mkdir(parents=True, exist_ok=True)
    print("Extracting (this takes a while — ~5.4GB, ~310k files)...")
    subprocess.run(
        f'unzip -q -o "{FULL_ZIP}" -d "/content/dataset/combined_resized" -x "__MACOSX/*" "*/._*"',
        shell=True, check=True
    )
    print("✅ Image data extracted")
else:
    print("✅ Image data already on disk, skipping extraction")

n_crop_folders = sum(1 for p in BASE.iterdir() if p.is_dir())
print(f"\nTop-level crop folders found: {n_crop_folders} (expect 33)")

Extracting combined_resized from Drive (split zip)...
✅ Copied zip parts to local disk
Recombining split zip parts...
✅ Recombined
Extracting (this takes a while — ~5.4GB, ~310k files)...
✅ Image data extracted

Top-level crop folders found: 33 (expect 33)


In [ ]:
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path

# crop-name aliases: subfolder names that redundantly repeat the crop name get that prefix stripped
CROP_ALIASES = {
    'bell_pepper': ['bell_pepper', 'pepperbell', 'pepper_bell'],
    'cherry': ['cherry'],
    'corn': ['corn'],
    'grape': ['grape'],
    'coffee': ['coffee'],
    'pineapple': ['pineapple'],
    'tomato': ['tomato'],
    'cotton': ['cotton'],
}

# exact overrides for names the automatic normalizer wouldn't get right (typos, ambiguous terms)
DISEASE_OVERRIDES = {
    ('soybean', 'Sudden Death Syndrone'): 'sudden_death_syndrome',
    ('soybean', 'Mossaic Virus'): 'mosaic_virus',
    ('soybean', 'Yellow Mosaic'): 'yellow_mosaic',
    ('soybean', 'ferrugen'): 'ferrugen_rust',
    ('groundnut', 'early_leaf_spot_1'): 'early_leaf_spot',
    ('groundnut', 'early_rust_1'): 'early_rust',
    ('groundnut', 'late_leaf_spot_1'): 'late_leaf_spot',
    ('groundnut', 'rust_1'): 'rust',
    ('groundnut', 'healthy_leaf_1'): 'healthy',
    ('groundnut', 'nutrition_deficiency_1'): 'nutrition_deficiency',
    ('onion', 'Stemphylium leaf blight and collectrichum leaf blight'): 'stemphylium_collectotrichum_blight',
    ('corn', 'Cercospora_leaf_spot Gray_leaf_spot'): 'cercospora_gray_leaf_spot',
    ('corn', 'Corn holcus_ leaf spot'): 'holcus_leaf_spot',
    ('corn', 'Common_rust_'): 'common_rust',
    ('tomato', 'Spider_mites Two-spotted_spider_mite'): 'spider_mites',
}

HEALTHY_KEYWORDS = {'healthy', 'normal', 'good_leaves', 'healthy_leaf', 'healthy_leaves'}


def normalize_disease_name(crop, raw_name):
    override = DISEASE_OVERRIDES.get((crop, raw_name))
    if override:
        return override

    s = raw_name.strip()
    s = s.replace('(', ' ').replace(')', ' ')
    s = re.sub(r'[^a-zA-Z0-9]+', '_', s)
    s = s.strip('_').lower()
    s = re.sub(r'_\d+$', '', s)  # strip trailing sample-count suffix, e.g. "..._220"

    for alias in sorted(CROP_ALIASES.get(crop, []), key=len, reverse=True):
        prefix = alias + '_'
        if s.startswith(prefix):
            s = s[len(prefix):]
            break

    if s in HEALTHY_KEYWORDS or 'healthy' in s or s == 'normal':
        return 'healthy'

    return s


# Build records from Stage 1's already-cleaned per-crop file lists
records = []
audit_rows = []

for crop in all_crops:
    ckpt = pd.read_csv(STAGE1_DEDUP_DIR / f'{crop}.csv')
    for _, row in ckpt.iterrows():
        parts = Path(row['filepath']).parts
        if len(parts) < 2:
            continue  # file sitting directly in crop folder, no disease subfolder — shouldn't happen but skip safely
        raw_disease = parts[0]  # filepath is relative to crop folder already in Stage 1 checkpoints? verify below
        disease_slug = normalize_disease_name(crop, raw_disease)
        label = f'{crop}__{disease_slug}'
        records.append({'filepath': row['filepath'], 'crop': crop, 'disease': label})

df = pd.DataFrame(records)
print(f"Total images: {len(df):,}")
print(f"Crops: {df['crop'].nunique()}")
print(f"Disease classes: {df['disease'].nunique()}")

NameError: name 'all_crops' is not defined

In [ ]:
raw_disease = parts[1]  # parts[0] = crop name, parts[1] = disease subfolder

NameError: name 'parts' is not defined

In [ ]:
records = []
for crop in all_crops:
    ckpt = pd.read_csv(STAGE1_DEDUP_DIR / f'{crop}.csv')
    for _, row in ckpt.iterrows():
        parts = Path(row['filepath']).parts
        if len(parts) < 3:  # expect crop/disease/filename at minimum
            continue
        raw_disease = parts[1]
        disease_slug = normalize_disease_name(crop, raw_disease)
        label = f'{crop}__{disease_slug}'
        records.append({'filepath': row['filepath'], 'crop': crop, 'disease': label})

df = pd.DataFrame(records)
print(f"Total images: {len(df):,}")
print(f"Crops: {df['crop'].nunique()}")
print(f"Disease classes: {df['disease'].nunique()}")

print(f"\nPer-crop disease breakdown:")
for crop in sorted(df['crop'].unique()):
    crop_df = df[df['crop'] == crop]
    diseases = sorted(crop_df['disease'].unique())
    print(f"\n  {crop} ({len(crop_df):,} images, {len(diseases)} classes):")
    for d in diseases:
        n = (crop_df['disease'] == d).sum()
        flag = '  ⚠️ small (<20)' if n < MIN_DISEASE_COUNT else ''
        print(f"    {d:<50} {n:>6,}{flag}")

NameError: name 'all_crops' is not defined

In [ ]:
def normalize_disease_name(crop, raw_name):
    override = DISEASE_OVERRIDES.get((crop, raw_name))
    if override:
        return override

    s = raw_name.strip()
    s = s.replace('(', ' ').replace(')', ' ')
    s = re.sub(r'[^a-zA-Z0-9]+', '_', s)
    s = s.strip('_').lower()
    s = re.sub(r'_\d+$', '', s)

    for alias in sorted(CROP_ALIASES.get(crop, []), key=len, reverse=True):
        prefix = alias + '_'
        if s.startswith(prefix):
            s = s[len(prefix):]
            break

    # broadened: catches "normal_leaf", "normal", "good_leaves", "healthy_leaf(s)", etc.
    if 'healthy' in s or 'normal' in s or s in {'good_leaves', 'good_leaf'}:
        return 'healthy'

    return s


records = []
for crop in all_crops:
    ckpt = pd.read_csv(STAGE1_DEDUP_DIR / f'{crop}.csv')
    for _, row in ckpt.iterrows():
        parts = Path(row['filepath']).parts
        if len(parts) < 3:
            continue
        raw_disease = parts[1]
        disease_slug = normalize_disease_name(crop, raw_disease)
        label = f'{crop}__{disease_slug}'
        records.append({'filepath': row['filepath'], 'crop': crop, 'disease': label})

df = pd.DataFrame(records)
print(f"Total images: {len(df):,}")
print(f"Crops: {df['crop'].nunique()}")
print(f"Disease classes: {df['disease'].nunique()}")

# quick check the fix worked
cherry_diseases = sorted(df[df['crop']=='cherry']['disease'].unique())
print(f"\ncherry classes now: {cherry_diseases}")
assert 'cherry__normal_leaf' not in cherry_diseases, "fix didn't take"
print("✅ merge confirmed")

NameError: name 'all_crops' is not defined

In [ ]:
def stratified_split(df, label_col='disease', train_frac=0.70, val_frac=0.15, seed=SEED):
    rng = np.random.default_rng(seed)
    train_rows, val_rows, test_rows = [], [], []
    for cls, group in df.groupby(label_col):
        idx = group.index.to_numpy().copy()
        rng.shuffle(idx)
        n = len(idx)
        n_train = max(1, int(round(n * train_frac)))
        n_val = max(1, min(int(round(n * val_frac)), n - n_train))
        train_rows.append(group.loc[idx[:n_train]])
        val_rows.append(group.loc[idx[n_train:n_train+n_val]])
        test_rows.append(group.loc[idx[n_train+n_val:]])
    return (pd.concat(train_rows).reset_index(drop=True),
            pd.concat(val_rows).reset_index(drop=True),
            pd.concat(test_rows).reset_index(drop=True))

train_df, val_df, test_df = stratified_split(df)

print(f"Train : {len(train_df):,}")
print(f"Val   : {len(val_df):,}")
print(f"Test  : {len(test_df):,}")

# every disease class must appear in all 3 splits — with min 20 images this should always hold, but verify
missing = (set(df['disease']) - set(train_df['disease'])) | \
          (set(df['disease']) - set(val_df['disease'])) | \
          (set(df['disease']) - set(test_df['disease']))
print(f"Classes missing from any split: {missing if missing else 'none ✅'}")

sorted_diseases = sorted(df['disease'].unique())
class_to_idx = {d: i for i, d in enumerate(sorted_diseases)}
NUM_CLASSES = len(class_to_idx)
print(f"\nNUM_CLASSES: {NUM_CLASSES}")

crop_disease_map = {}
for crop in sorted(df['crop'].unique()):
    diseases = sorted(df[df['crop'] == crop]['disease'].unique().tolist())
    crop_disease_map[crop] = diseases

train_df.to_csv(DRIVE_ROOT / f'disease_train_manifest_{VERSION}.csv', index=False)
val_df.to_csv(DRIVE_ROOT / f'disease_val_manifest_{VERSION}.csv', index=False)
test_df.to_csv(DRIVE_ROOT / f'disease_test_manifest_{VERSION}.csv', index=False)

with open(MODEL_DIR / f'crop_disease_map_{VERSION}.json', 'w') as f:
    json.dump(crop_disease_map, f, indent=2)
with open(MODEL_DIR / f'class_to_idx_{VERSION}.json', 'w') as f:
    json.dump(class_to_idx, f, indent=2)

print(f"\n✅ Saved to Drive:")
print(f"  disease_train_manifest_{VERSION}.csv  ({len(train_df):,} rows)")
print(f"  disease_val_manifest_{VERSION}.csv    ({len(val_df):,} rows)")
print(f"  disease_test_manifest_{VERSION}.csv   ({len(test_df):,} rows)")
print(f"  crop_disease_map_{VERSION}.json  ({len(crop_disease_map)} crops)")
print(f"  class_to_idx_{VERSION}.json      ({NUM_CLASSES} disease classes)")

Train : 182,057
Val   : 39,014
Test  : 39,023
Classes missing from any split: none ✅

NUM_CLASSES: 203

✅ Saved to Drive:
  disease_train_manifest_v2.csv  (182,057 rows)
  disease_val_manifest_v2.csv    (39,014 rows)
  disease_test_manifest_v2.csv   (39,023 rows)
  crop_disease_map_v2.json  (33 crops)
  class_to_idx_v2.json      (203 disease classes)


In [ ]:
!pip install -q albumentations

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

train_transform = A.Compose([
    A.Resize(256, 256, interpolation=cv2.INTER_CUBIC),
    A.RandomCrop(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=20, p=0.5, border_mode=cv2.BORDER_REFLECT),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20),
        A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15),
    ], p=0.7),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5)),
        A.GaussNoise(),
        A.ImageCompression(quality_range=(40, 95)),
    ], p=0.3),
    A.CoarseDropout(
        num_holes_range=(1, 4),
        hole_height_range=(8, 20),
        hole_width_range=(8, 20),
        p=0.25,
    ),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(256, 256, interpolation=cv2.INTER_CUBIC),
    A.CenterCrop(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])

print('✅  Transforms built: train_transform, val_transform')

✅  Transforms built: train_transform, val_transform


In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path

with open(MODEL_DIR / f'class_to_idx_{VERSION}.json') as f:
    class_to_idx = json.load(f)
with open(MODEL_DIR / f'crop_disease_map_{VERSION}.json') as f:
    crop_disease_map = json.load(f)

NUM_CLASSES = len(class_to_idx)
idx_to_class = {v: k for k, v in class_to_idx.items()}
print(f'Loaded: {NUM_CLASSES} disease classes across {len(crop_disease_map)} crops')


class DiseaseDataset(Dataset):
    """Same missing/corrupt-file resilience pattern as Stage 1's CropDataset —
    a bad file logs a warning and gets skipped via safe_collate, never crashes an epoch."""

    def __init__(self, csv_path, data_root, class_to_idx, transform, split='train'):
        self.df = pd.read_csv(csv_path)
        self.data_root = Path(data_root)
        self.class_to_idx = class_to_idx
        self.transform = transform
        self.split = split
        self._missing_logged = 0

        unknown = set(self.df['disease'].unique()) - set(class_to_idx)
        assert not unknown, f'Unknown disease labels in manifest: {unknown}'

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fpath = self.data_root / row['filepath']

        if not fpath.exists():
            self._missing_logged += 1
            if self._missing_logged <= 20:
                print(f"⚠️  [{self.split}] missing file, skipping: {row['filepath']}")
            return None

        try:
            img = np.array(Image.open(fpath).convert('RGB'))
        except Exception as e:
            print(f"⚠️  [{self.split}] unreadable file, skipping: {row['filepath']} ({e})")
            return None

        img_t = self.transform(image=img)['image']
        label_idx = self.class_to_idx[row['disease']]
        return img_t, label_idx


def safe_collate(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    imgs, labels = zip(*batch)
    return torch.stack(imgs), torch.tensor(labels)


train_ds = DiseaseDataset(DRIVE_ROOT / f'disease_train_manifest_{VERSION}.csv', BASE, class_to_idx, train_transform, split='train')
val_ds   = DiseaseDataset(DRIVE_ROOT / f'disease_val_manifest_{VERSION}.csv',   BASE, class_to_idx, val_transform,   split='val')
test_ds  = DiseaseDataset(DRIVE_ROOT / f'disease_test_manifest_{VERSION}.csv',  BASE, class_to_idx, val_transform,   split='test')

# sqrt-inverse-frequency, normalized to mean=1 — same approach as Stage 1, much more stable
# than raw inverse-frequency given the ~1,400:1 imbalance ratio in this dataset
train_manifest_df = pd.read_csv(DRIVE_ROOT / f'disease_train_manifest_{VERSION}.csv')
class_counts = train_manifest_df['disease'].map(class_to_idx).value_counts().reindex(range(NUM_CLASSES), fill_value=1)
sqrt_inv_freq = 1.0 / np.sqrt(class_counts.to_numpy())
weights = torch.tensor(sqrt_inv_freq / sqrt_inv_freq.mean(), dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTHING)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, collate_fn=safe_collate)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True, collate_fn=safe_collate)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True, collate_fn=safe_collate)

batch = next(iter(train_loader))
print(f'\n✅  DataLoaders ready')
print(f'    Train        : {len(train_ds):,} samples')
print(f'    Val          : {len(val_ds):,} samples')
print(f'    Test         : {len(test_ds):,} samples')
if batch is not None:
    imgs, labels = batch
    print(f'    Batch shape  : {tuple(imgs.shape)}')
print(f'    Steps/epoch  : {len(train_loader)}')
print(f'    Class weight range: {weights.min():.3f} – {weights.max():.3f}')

Loaded: 203 disease classes across 33 crops

✅  DataLoaders ready
    Train        : 182,057 samples
    Val          : 39,014 samples
    Test         : 39,023 samples
    Batch shape  : (64, 3, 224, 224)
    Steps/epoch  : 2844
    Class weight range: 0.141 – 5.298


In [ ]:
import timm

model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES, drop_rate=DROPOUT)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'✅  Model built: {MODEL_NAME}')
print(f'    Num classes      : {NUM_CLASSES}')
print(f'    Total params     : {total_params/1e6:.2f}M')
print(f'    Trainable params : {trainable_params/1e6:.2f}M')

model.eval()
with torch.no_grad():
    batch = next(iter(train_loader))
    imgs, labels = batch
    imgs = imgs.to(device)
    out = model(imgs)

assert out.shape == (imgs.shape[0], NUM_CLASSES), f'Unexpected output shape: {out.shape}'
print(f'    Forward pass OK  : input {tuple(imgs.shape)} → output {tuple(out.shape)}')
assert NUM_CLASSES == len(class_to_idx)
print(f'    ✅  NUM_CLASSES matches class_to_idx.json ({len(class_to_idx)} classes)')

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

✅  Model built: efficientnet_b0
    Num classes      : 203
    Total params     : 4.27M
    Trainable params : 4.27M
    Forward pass OK  : input (64, 3, 224, 224) → output (64, 203)
    ✅  NUM_CLASSES matches class_to_idx.json (203 classes)


In [ ]:
import time, json
from datetime import datetime
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm

CKPT_PATH = MODEL_DIR / 'latest_checkpoint.pt'
BEST_CKPT_PATH = MODEL_DIR / 'best_checkpoint.pt'
F1_HISTORY_PATH = MODEL_DIR / 'f1_history.json'


def evaluate(loader):
    model.eval()
    total_loss, all_preds, all_labels, n = 0.0, [], [], 0
    with torch.no_grad():
        for batch in loader:
            if batch is None:
                continue
            imgs, labels = batch
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                logits = model(imgs)
                loss = criterion(logits, labels)
            total_loss += loss.item() * imgs.size(0)
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            n += imgs.size(0)
    return (
        total_loss / max(n, 1),
        accuracy_score(all_labels, all_preds),
        f1_score(all_labels, all_preds, average='macro', zero_division=0),
    )


def save_checkpoint(path, epoch, best_f1):
    tmp = path.with_suffix('.pt.tmp')
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'best_f1': best_f1,
        'class_to_idx': class_to_idx,
    }, tmp)
    tmp.replace(path)

def load_f1_history():
    if F1_HISTORY_PATH.exists():
        with open(F1_HISTORY_PATH) as f:
            return json.load(f)
    return []

def save_f1_history(history):
    tmp = F1_HISTORY_PATH.with_suffix('.json.tmp')
    with open(tmp, 'w') as f:
        json.dump(history, f, indent=2)
    tmp.replace(F1_HISTORY_PATH)


optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

start_epoch, best_f1 = 1, -1.0
if CKPT_PATH.exists():
    print(f"Found checkpoint — resuming")
    ckpt = torch.load(CKPT_PATH, map_location=device)
    assert ckpt['class_to_idx'] == class_to_idx, "❌ checkpoint class_to_idx mismatch — do not resume blindly"
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch = ckpt['epoch'] + 1
    best_f1 = ckpt['best_f1']
    print(f"  Resumed at epoch {start_epoch}, best F1 so far {best_f1:.4f}")
else:
    print("No checkpoint found — starting fresh")

history = load_f1_history()

print(f'\nTraining epochs {start_epoch}-{EPOCHS} | {len(train_loader)} steps/epoch | batch {BATCH_SIZE}')
print(f"{'Ep':>4}  {'TrLoss':>8}  {'TrAcc':>7}  {'VLoss':>8}  {'VAcc':>7}  {'M-F1':>7}  {'LR':>9}")
print('─' * 72)

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    run_loss, correct, total = 0.0, 0, 0
    t0 = time.time()

    for batch in tqdm(train_loader, desc=f'Ep {epoch}/{EPOCHS}', leave=False):
        if batch is None:
            continue
        imgs, labels = batch
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits = model(imgs)
            loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        run_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    scheduler.step()

    tr_loss, tr_acc = run_loss / max(total, 1), correct / max(total, 1)
    v_loss, v_acc, v_mf1 = evaluate(val_loader)
    cur_lr = scheduler.get_last_lr()[0]
    elapsed = time.time() - t0

    save_checkpoint(CKPT_PATH, epoch, best_f1)
    improved = v_mf1 > best_f1
    if improved:
        best_f1 = v_mf1
        save_checkpoint(BEST_CKPT_PATH, epoch, best_f1)

    flag = '✅' if improved else ''
    print(f'{epoch:>4}  {tr_loss:>8.4f}  {tr_acc:>7.4f}  {v_loss:>8.4f}  '
          f'{v_acc:>7.4f}  {v_mf1:>7.4f}  {cur_lr:>9.2e}  {flag}  ({elapsed:.0f}s)')

    history.append({'epoch': epoch, 'tr_loss': tr_loss, 'tr_acc': tr_acc,
                     'v_loss': v_loss, 'v_acc': v_acc, 'v_macro_f1': v_mf1,
                     'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M')})
    save_f1_history(history)

print(f'\n✅  Best val macro F1: {best_f1:.4f}  →  {BEST_CKPT_PATH}')

No checkpoint found — starting fresh

Training epochs 1-20 | 2844 steps/epoch | batch 64
  Ep    TrLoss    TrAcc     VLoss     VAcc     M-F1         LR
────────────────────────────────────────────────────────────────────────


Ep 1/20:   0%|          | 0/2844 [00:00<?, ?it/s]

   1    2.7590   0.8490    2.9793   0.9250   0.7948   9.94e-04  ✅  (368s)


Ep 2/20:   0%|          | 0/2844 [00:00<?, ?it/s]

   2    2.4290   0.9116    2.9199   0.9450   0.8356   9.76e-04  ✅  (366s)


...

  18    2.0215   0.9907    2.7411   0.9877   0.9549   2.54e-05  ✅  (367s)


Ep 19/20:   0%|          | 0/2844 [00:00<?, ?it/s]

  19    2.0197   0.9912    2.7416   0.9869   0.9525   7.15e-06    (367s)


Ep 20/20:   0%|          | 0/2844 [00:00<?, ?it/s]

  20    2.0169   0.9916    2.7404   0.9873   0.9520   1.00e-06    (367s)

✅  Best val macro F1: 0.9549  →  /content/drive/MyDrive/CropClassifier_v2/models/stage2_disease_v2/best_checkpoint.pt
